# ==============================================================================
# 📡 TELECOMMUNICATIONS: CUSTOMER CHURN VIA ADVANCED ADABOOST CLASSIFIER
# ==============================================================================
### Core Principles:
Customer retention management requires identifying at-risk accounts before service cancellation.
Integrates `ColumnTransformer` encoding multi-attribute contracts with AdaBoost sequential boosting.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Telecom Churn Environment ready.")


✅ STEP 1: Telecom Churn Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Telco Customer Churn dataset (7,043 customer accounts).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/customer_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data_path)

if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

df.columns = [c.strip().lower() for c in df.columns]

# Totalcharges has whitespace string nulls
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')
df['totalcharges'] = df['totalcharges'].fillna(df['totalcharges'].median())

print(f"📊 Dataset Shape: {df.shape[0]} customers, {df.shape[1]} attributes")
print(f"Churn Rate:\n{df['churn'].value_counts(normalize=True).round(4)*100}%")
df.head(3)


📊 Dataset Shape: 7043 customers, 20 attributes
Churn Rate:
churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64%


,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `churn` ('Yes' / 1, 'No' / 0).


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['churn'])
y = (df['churn'].str.strip().str.lower() == 'yes').astype(int)

print(f"Features: {X.shape[1]}")
print(f"Missing values: {X.isnull().sum().sum()}")


Features: 19
Missing values: 0


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
80/20 stratified split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Cohort: {X_train.shape[0]} ({y_train.sum()} Churned)")
print(f"Test Cohort : {X_test.shape[0]} ({y_test.sum()} Churned)")


Train Cohort: 5634 (1495 Churned)
Test Cohort : 1409 (374 Churned)


## ⚙️ STEP 7: ENTERPRISE PREPROCESSOR PIPELINE
Encoding categorical subscription packages alongside continuous billing variables.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])
print("✅ STEP 7: ColumnTransformer Preprocessor configured.")


✅ STEP 7: ColumnTransformer Preprocessor configured.


/tmp/ipykernel_692846/4030336783.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


## ⚠️ STEP 8: BASELINE PIPELINE FIT
Fitting baseline AdaBoost with 50 estimators.


In [6]:
# STEP 8: BASELINE FIT
pipe_ada = Pipeline([
    ('prep', preprocessor),
    ('ada', AdaBoostClassifier(n_estimators=50, random_state=42))
])
pipe_ada.fit(X_train, y_train)

acc_base = accuracy_score(y_test, pipe_ada.predict(X_test))
roc_base = roc_auc_score(y_test, pipe_ada.predict_proba(X_test)[:, 1])

print(f"Baseline AdaBoost Accuracy: {acc_base * 100:.2f}%")
print(f"Baseline AdaBoost ROC-AUC : {roc_base * 100:.2f}%")


Baseline AdaBoost Accuracy: 80.55%
Baseline AdaBoost ROC-AUC : 84.16%


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing `n_estimators` and `learning_rate` shrinkage.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'ada__n_estimators': [50, 100, 200],
    'ada__learning_rate': [0.05, 0.1, 0.5, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_ada, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Churn Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV ROC-AUC: {grid.best_score_ * 100:.2f}%")


🏆 Best Churn Hyperparameters: {'ada__learning_rate': 1.0, 'ada__n_estimators': 200}
🎯 Best 5-Fold CV ROC-AUC: 84.87%


## 📊 STEP 10 & 11: RETENTION REPORT & CONFUSION MATRIX
Reviewing business retention metrics and churn capture.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print("📋 TELECOM RETENTION PERFORMANCE REPORT:")
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

print(f"🎯 Test Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 ROC-AUC Score: {roc_auc_score(y_test, y_prob)*100:.2f}%")
print(f"🎯 Churn F1-Score: {f1_score(y_test, y_pred)*100:.2f}%")

print(f"\nConfusion Matrix:\n{cm}")


📋 TELECOM RETENTION PERFORMANCE REPORT:
              precision    recall  f1-score   support

    Retained       0.84      0.91      0.87      1035
     Churned       0.67      0.52      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409

🎯 Test Accuracy: 80.48%
🎯 ROC-AUC Score: 84.31%
🎯 Churn F1-Score: 58.77%

Confusion Matrix:
[[938  97]
 [178 196]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & RETENTION SMOKE TEST
Deploying pipeline and validating CRM account retention screening latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/telco_churn_adaboost_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_cust = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_churn = loaded_pipe.predict(sample_cust)[0]
pred_prob = loaded_pipe.predict_proba(sample_cust)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n📡 LIVE CRM ACCOUNT RETENTION AUDIT:")
print(f"   Account Risk Status: {'🚨 HIGH CHURN RISK' if pred_churn == 1 else '🟢 STABLE / RETAINED'}")
print(f"   Churn Probability  : {pred_prob*100:.2f}%")
print(f"   CRM Screening Time : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/telco_churn_adaboost_pipeline.joblib (24,744 bytes)



📡 LIVE CRM ACCOUNT RETENTION AUDIT:
   Account Risk Status: 🟢 STABLE / RETAINED
   Churn Probability  : 31.50%
   CRM Screening Time : 103.556 ms (SLA < 2.0ms: CHECK)
